In [1]:
import os
import math
import torch
import tiktoken
import numpy as np 
import torch.nn as nn
from PIL import Image
import torch.nn.functional as F
from torchvision.transforms import v2 
from dataclasses import dataclass
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10, CIFAR100, OxfordIIITPet, Food101

from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')

In [2]:
#defualt is highest
torch.set_float32_matmul_precision('high')

In [3]:
@dataclass
class Configuration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 256
    num_workers: int = 10
    device: str = 'cuda:0'  
    model_dir: str = 'models_clip_cc3m'
    model_name : str = 'clip_base_patch16_224_14.pth'
    
    #vit model hyperparameters
    img_size: int = 224
    patch_size: int = 16
    vit_dim: int = 768 
    vit_heads: int = 12 
    vit_encoder_layers: int = 12 
    dropout: float = 0.1

    #transformer hyperparameters
    context_length: int = 77
    transformer_width: int = 512
    transformer_heads: int = 8
    transformer_layers: int = 12

    projection_dim: int = 512

    mscoco_captions_path: str = '../datasets/mscoco/captions_val2017.json'
    mscoco_images_path: str = '../datasets/mscoco/val2017/'


In [4]:
class FFN(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
    def forward(self,x):
        return self.feed_forward(x)

In [5]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.h = h
        
        assert d_model%h ==0, "d_model should be divisible by h"
        
        self.d_k = d_model//h
        self.w_q = nn.Linear(d_model,d_model, bias= False)
        self.w_k = nn.Linear(d_model,d_model, bias= False)        
        self.w_v = nn.Linear(d_model, d_model, bias= False)  
        
        self.w_o = nn.Linear(d_model, d_model, bias= False)  
        self.dropout = nn.Dropout(dropout)
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        scores = query@key.transpose(-2,-1)/math.sqrt(d_k)
        #print(query.shape,key.shape, value.shape, scores.shape, mask.shape)
        if mask is not None:
            scores.masked_fill_(mask==0, -1e9)
        scores = scores.softmax(dim=-1)
        
        ### do we need ?
        if dropout is not None:
            scores = dropout(scores)
            
        return scores@value,scores 
        
    def forward(self, q, k, v, mask):
        # batch, seq len, embed dim
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)
        
        # batch, h, seq, d_k
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1,2)
        
        #x, scores = self.attention(query, key, value, mask, self.dropout)
        #using inbuilt flash attention
        x = F.scaled_dot_product_attention(query, key, value, attn_mask=mask, dropout_p=self.dropout.p)

        x = x.transpose(1,2).contiguous().view(x.shape[0], -1, self.h*self.d_k)
        
        return self.w_o(x)
        

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model:int, h:int, d_ff:int, dropout:float):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, h, dropout)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.layer_norm_1 = nn.LayerNorm(d_model)
        self.layer_norm_2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        x_ln = self.layer_norm_1(x)
        attention = self.attention(x_ln, x_ln, x_ln, mask)
        x = x + self.dropout(attention)
        
        ffn_out = self.ffn(self.layer_norm_2(x))
        x = x+ self.dropout(ffn_out)
        
        return x
        

In [7]:
class Transformer(nn.Module):
    def __init__(self,d_model:int, h:int, d_ff:int, dropout:float, N=6):
        super().__init__()
        
        self.encoder = nn.ModuleList(
            [
                EncoderBlock(d_model, h, d_ff, dropout)
                for _ in range(N)
            ]
        )
        
    
    def forward(self,x, mask=None):
        for layer in  self.encoder:
            x = layer(x, mask)
        return x
    

In [8]:
def img_to_patches(img, patch_size):
    bs, c, h, w = img.shape
    assert h % patch_size == 0 and w % patch_size == 0, "Image dimensions must be divisible by the patch size."
    num_patches_h = h // patch_size
    num_patches_w = w // patch_size
    img = img.reshape(bs, c, num_patches_h, patch_size, num_patches_w, patch_size)
    img = img.permute(0, 2, 4, 1, 3, 5).contiguous()  # (batch_size, num_patches_h, num_patches_w, channels, patch_size, patch_size)
    patches = img.view(bs, num_patches_h * num_patches_w, c * patch_size * patch_size)  # (batch_size, num_patches, patch_dim)
    return patches


In [9]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, projection_dim=512,
                  dim =768, heads=8, encoder_layers=12, dropout=0.1):
        super(ViT, self).__init__() 
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.linear_proj = nn.Linear(3*patch_size*patch_size, dim)
        self.transformer = Transformer(dim, heads, dim*4, dropout, encoder_layers)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.embedding = nn.Parameter(torch.randn(1, self.num_patches+1, dim))
        self.dropout = nn.Dropout(dropout)
        self.ln_pre = nn.LayerNorm(dim)
        self.ln_post = nn.LayerNorm(dim)
        self.projection_layer = nn.Parameter(torch.randn(dim, projection_dim))

    def forward(self, x):
        x = img_to_patches(x, self.patch_size)  # (batch_size, num_patches, patch_dim)
        x = self.linear_proj(x)  # (batch_size, num_patches, dim)
        cls_token = self.cls_token.repeat(x.shape[0], 1, 1)  # (batch_size, 1, dim)
        x = torch.cat((cls_token, x), dim=1)  # (batch_size, num_patches + 1, patch_dim)
        x = x + self.embedding  # (batch_size, num_patches + 1, dim)
        x = self.dropout(x)
        x = self.ln_pre(x)
        x = self.transformer(x)  # (batch_size, num_patches + 1, dim)
        x = self.ln_post(x)
        x = x[:, 0]  # (batch_size, dim)
        x = torch.matmul(x, self.projection_layer)
        return x

In [10]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, context_length, h,
                  d_ff, dropout, N, projection_dim):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = nn.Parameter(torch.randn(1, context_length, d_model))
        self.transformer = Transformer(d_model, h, d_ff, dropout, N)
        self.ln_post = nn.LayerNorm(d_model)
        self.projection_layer = nn.Parameter(torch.randn(d_model, projection_dim))

    def forward(self, x):
        masked = x.clone()
        masked[:,0]=-1
        pos = masked.argmax(dim=-1)

        x = self.token_embedding(x)  # (batch_size, seq_len, d_model)
        x = x + self.positional_embedding[:, :x.shape[1], :]  # (batch_size, seq_len, d_model)
        x = self.transformer(x)  # (batch_size, seq_len, d_model)
        x = self.ln_post(x)
        x = x[torch.arange(x.shape[0]), pos, :]  # (batch_size, d_model)
        x = torch.matmul(x, self.projection_layer)  # (batch_size, projection_dim)
        return x

In [11]:
class CLIP(nn.Module):
    def __init__(self, config: Configuration, tokenizer: tiktoken.core.Encoding):
        super().__init__()
        self.image_encoder = ViT(
            img_size=config.img_size,
            patch_size=config.patch_size,
            projection_dim=config.projection_dim,
            dim =config.vit_dim,
            heads=config.vit_heads,
            encoder_layers=config.vit_encoder_layers,
            dropout=config.dropout
        )
        self.text_encoder = TextEncoder(
            vocab_size=tokenizer.n_vocab,
            d_model=config.transformer_width,
            context_length=config.context_length,
            h=config.transformer_heads,
            d_ff=config.transformer_width*4,
            dropout=config.dropout,
            N=config.transformer_layers,
            projection_dim=config.projection_dim,
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))


        #self.initialize_weights()
    def encode_image(self, images):
        image_features = self.image_encoder(images)  # (batch_size, projection_dim)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features

    def encode_text(self, tokens):
        text_features = self.text_encoder(tokens)  # (batch_size, projection_dim)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features

    def forward(self, images, tokens):
        image_features = self.encode_image(images)
        text_features = self.encode_text(tokens)

        return image_features, text_features


In [12]:
tokenizer = tiktoken.get_encoding('gpt2')

In [13]:
def tokenize(txt, tokenizer=tokenizer,context_length=77):
    '''
        GPT2 don't have sos and pad tokens but we 
        can use eot token as both sos and pad token
    '''
    eos_token_id = tokenizer.eot_token
    sos_token_id = eos_token_id
    pad_token_id = eos_token_id

    tokens = tokenizer.encode(txt)
    tokens = tokens[:context_length - 2]
    tokens = [sos_token_id] + tokens + [eos_token_id]
    tokens += [pad_token_id] * (context_length - len(tokens))

    return np.array(tokens, dtype=np.int32)

In [14]:
def transform():
    return v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize((224, 224)),
        v2.ToDtype(torch.float32, scale=True),  # Normalize expects float input
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [15]:
def get_tokenized_captions(prompts):
    tokenized_captions = np.array([tokenize(prompt) for prompt in prompts])
    return torch.tensor(tokenized_captions)

In [16]:
config = Configuration()

clip_model = CLIP(config, tokenizer)
clip_model = torch.compile(clip_model)
clip_model.to(config.device)

states =  torch.load(os.path.join(config.model_dir, config.model_name), map_location=config.device)
clip_model.load_state_dict(states['model_state_dict'])

clip_model.eval()
print(' ')

In [17]:
##COCO retrieval evaluation

from pycocotools.coco import COCO

def coco_retrieval_eval(model, device, 
                         ann_file= config.mscoco_captions_path,
                         img_dir= config.mscoco_images_path):
    coco = COCO(ann_file)
    img_ids = list(coco.imgs.keys())[:5000]  # use 5k images

    # encode all images
    img_embs = []
    for img_id in img_ids:
        path = os.path.join(img_dir, coco.imgs[img_id]["file_name"])
        image = Image.open(path).convert("RGB")
        image = transform()(image).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = model.image_encoder(image)
        img_embs.append(emb)
    img_embs = torch.cat(img_embs, dim=0)  # (5000, 512)

    # encode first caption per image
    txt_embs = []
    for img_id in img_ids:
        ann_ids = coco.getAnnIds(imgIds=img_id)
        caption = coco.loadAnns(ann_ids)[0]["caption"]
        tokens = tokenize(caption)
        tokens    = torch.tensor(tokens).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = model.text_encoder(tokens)
        txt_embs.append(emb)
    txt_embs = torch.cat(txt_embs, dim=0)  # (5000, 512)

    # text-to-image R@1, R@5, R@10
    sims = txt_embs @ img_embs.T  # (5000, 5000)
    for k in [1, 5, 10]:
        topk = sims.topk(k, dim=1).indices
        hits = sum(topk[i].tolist().__contains__(i) for i in range(len(img_ids)))
        print(f"T2I R@{k}: {hits/len(img_ids)*100:.2f}%")


    # I2T: given an image, retrieve the correct caption
    sims = img_embs @ txt_embs.T   # (5000, 5000)
    for k in [1, 5, 10]:
        topk = sims.topk(k, dim=1).indices
        hits = sum(topk[i].tolist().__contains__(i) 
                for i in range(len(img_ids)))
        print(f"I2T R@{k}: {hits/len(img_ids)*100:.2f}%")

In [18]:
coco_retrieval_eval(clip_model, config.device)

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
T2I R@1: 8.82%
T2I R@5: 22.38%
T2I R@10: 31.48%
I2T R@1: 5.84%
I2T R@5: 17.90%
I2T R@10: 27.06%


In [19]:
def evaluate_zero_shot(config, clip_model, dataset, prompts):
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers)
    tokenized_captions = get_tokenized_captions(prompts)
    with torch.no_grad():
        text_features = clip_model.text_encoder(tokenized_captions.to(config.device))
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)


        correct = total = 0
        for images, labels in loader:
            images = images.to(config.device)
            img_features = clip_model.image_encoder(images)     # (B, 512)
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)  # (B, 512)
            sims = img_features @ text_features.T              # (B, 10)
            preds = sims.argmax(dim=-1)
            correct += (preds.cpu() == labels).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    return accuracy

In [20]:
dataset = CIFAR10(root='../datasets', train=False, download=True, transform=transform())
prompts = [f"a photo of a {c}" for c in dataset.classes]

zs_accuracy_cifar10 = evaluate_zero_shot(config, clip_model, dataset, prompts)

print(f"Zero-Shot Accuracy on CIFAR-10: {zs_accuracy_cifar10*100:.2f}%")

Zero-Shot Accuracy on CIFAR-10: 47.68%


In [21]:
dataset = CIFAR100(root='../datasets', train=False, download=True, transform=transform())
prompts = [f"a photo of a {c}" for c in dataset.classes]

zs_accuracy_cifar100 = evaluate_zero_shot(config, clip_model, dataset, prompts)

print(f"Zero-Shot Accuracy on CIFAR-100: {zs_accuracy_cifar100*100:.2f}%")

Zero-Shot Accuracy on CIFAR-100: 23.68%


In [22]:
dataset = OxfordIIITPet(root='../datasets', split='test', download=True, transform=transform())
prompts = [f"a photo of a {c}, a type of a pet." for c in dataset.classes]
zs_accuracy_oxford = evaluate_zero_shot(config, clip_model, dataset, prompts)

print(f"Zero-Shot Accuracy on Oxford-IIIT-Pet: {zs_accuracy_oxford*100:.2f}%")

Zero-Shot Accuracy on Oxford-IIIT-Pet: 5.78%


In [23]:
dataset = Food101(root='../datasets', split='test', download=True, transform=transform())
# some Food101 class names have underscores
# "apple_pie" should become "apple pie"
classes = [c.replace("_", " ") for c in dataset.classes]
prompts = [f"a photo of a {c}, a type of a food." for c in classes]
zs_accuracy_food101 = evaluate_zero_shot(config, clip_model, dataset, prompts)

print(f"Zero-Shot Accuracy on Food101: {zs_accuracy_food101*100:.2f}%")

Zero-Shot Accuracy on Food101: 8.34%


In [24]:
def get_features(dataset):
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in DataLoader(dataset, batch_size=100):
            features = clip_model.image_encoder(images.to(config.device))

            all_features.append(features)
            all_labels.append(labels)

    return torch.cat(all_features).cpu().numpy(), torch.cat(all_labels).cpu().numpy()

In [25]:
def linear_probe(train_dataset, test_dataset):
    train_features, train_labels = get_features(train_dataset)
    test_features, test_labels = get_features(test_dataset)

    classifier = LogisticRegression(random_state=17,max_iter=1000, verbose=1)
    classifier.fit(train_features, train_labels)

    predictions = classifier.predict(test_features)
    accuracy = np.mean((test_labels == predictions).astype(float))
    
    return accuracy

In [26]:
train = CIFAR10(root = '../datasets', download=True, train=True, transform=transform())
test = CIFAR10(root = '../datasets', download=True, train=False, transform=transform())

lp_accuracy_cifar10 = linear_probe(train, test)
print(f"Linear Probe Accuracy on CIFAR-10 = {lp_accuracy_cifar10*100:.2f}%")

print('Increase in performance from Zero-Shot to Linear Probe = {:.2f}%'.format((lp_accuracy_cifar10 - zs_accuracy_cifar10)*100))

Linear Probe Accuracy on CIFAR-10 = 78.76%
Increase in performance from Zero-Shot to Linear Probe = 31.08%


In [27]:
train = CIFAR100(root = '../datasets', download=True, train=True, transform=transform())
test = CIFAR100(root = '../datasets', download=True, train=False, transform=transform())

lp_accuracy_cifar100 = linear_probe(train, test)
print(f"Linear Probe Accuracy on CIFAR-100 = {lp_accuracy_cifar100*100:.2f}%")

print('Increase in performance from Zero-Shot to Linear Probe = {:.2f}%'.format((lp_accuracy_cifar100 - zs_accuracy_cifar100)*100))

Linear Probe Accuracy on CIFAR-100 = 56.22%
Increase in performance from Zero-Shot to Linear Probe = 32.54%


In [28]:
train = OxfordIIITPet(root='../datasets', split='trainval', download=True, transform=transform())
test = OxfordIIITPet(root='../datasets', split='test', download=True, transform=transform())

lp_accuracy_oxfordiiitpet = linear_probe(train, test)
print(f"Linear Probe Accuracy on OxfordIIITPet = {lp_accuracy_oxfordiiitpet*100:.2f}%")

print('Increase in performance from Zero-Shot to Linear Probe = {:.2f}%'.format((lp_accuracy_oxfordiiitpet - zs_accuracy_oxford)*100))

Linear Probe Accuracy on OxfordIIITPet = 49.36%
Increase in performance from Zero-Shot to Linear Probe = 43.58%


In [29]:
train = Food101(root='../datasets', split='train', download=True, transform=transform())
test = Food101(root='../datasets', split='test', download=True, transform=transform())

lp_accuracy_food101 = linear_probe(train, test)
print(f"Linear Probe Accuracy on Food101 = {lp_accuracy_food101*100:.2f}%")

print('Increase in performance from Zero-Shot to Linear Probe = {:.2f}%'.format((lp_accuracy_food101 - zs_accuracy_food101)*100))

Linear Probe Accuracy on Food101 = 44.21%
Increase in performance from Zero-Shot to Linear Probe = 35.87%
